<a href="https://colab.research.google.com/github/vchandraiitk/agentic-ai/blob/main/AgentLoop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Install LiteLLM (skip if already installed)
!pip install litellm openai --quiet

In [7]:

import os
from google.colab import userdata
import os
import json
import litellm
from typing import List

# 🔐 Setup LiteLLM
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')  # Replace with your real key
litellm.set_verbose = False
llm_model = "openai/gpt-4o"

# 🧠 Agent Instructions
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:
- list_files() -> List[str]: List all files in the current directory.
- read_file(file_name: str) -> str: Read the content of a file.
- terminate(message: str): End the agent loop and print a summary to the user.

If a user asks about files, list them before reading.

Every response MUST have an action.
Respond in this format:

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
"""
}]

# 🛠️ Tools
# prompt: I want to list all files recursivily and dont change function signature def list_files() -> List[str]:
#     return os.listdir()

def list_files() -> List[str]:
    all_files = []
    for root, _, files in os.walk("."):
        for file in files:
            all_files.append(os.path.join(root, file))
    return all_files


def read_file(file_name: str) -> str:
    try:
        with open(file_name, 'r') as f:
            return f.read()
    except Exception as e:
        return f"Error: {str(e)}"

# 🎯 Action Parser
def parse_action(response: str):
    try:
        start = response.find("```action")
        end = response.find("```", start + 1)
        json_str = response[start + 9:end].strip()
        return json.loads(json_str)
    except:
        return {"tool_name": "error", "args": {"message": "Invalid format or no action found."}}

# 🔁 LLM Response Generator
def generate_response(prompt):
    return litellm.completion(
        model=llm_model,
        messages=prompt,
        temperature=0.7
    )["choices"][0]["message"]["content"]

# 🧠 Memory Initialization
memory = []
max_iterations = 10
iterations = 0

# 🔁 Main Agent Loop
while iterations < max_iterations:
    user_input = input("🧑‍💻 You: ")
    memory.append({"role": "user", "content": user_input})

    print("\n🤖 Agent thinking...")
    prompt = agent_rules + memory
    response = generate_response(prompt)
    print(f"\n📨 Agent Response:\n{response}")

    action = parse_action(response)
    result = {}

    if action["tool_name"] == "list_files":
        result = {"result": list_files()}
    elif action["tool_name"] == "read_file":
        result = {"result": read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "terminate":
        print(f"\n✅ Terminated: {action['args']['message']}")
        break
    elif action["tool_name"] == "error":
        result = {"error": action["args"]["message"]}
    else:
        result = {"error": "Unknown action: " + action["tool_name"]}

    print(f"\n📦 Action Result: {result}")

    memory.append({"role": "assistant", "content": response})
    memory.append({"role": "user", "content": json.dumps(result)})

    iterations += 1


🧑‍💻 You: terminate kill it

🤖 Agent thinking...

📨 Agent Response:
```action
{
    "tool_name": "terminate",
    "args": {"message": "The process has been terminated as per your request."}
}
```

✅ Terminated: The process has been terminated as per your request.
